# SpkIOSzero — DRAGON BALL: Sparking! ZERO

Notebook com a mesma organização, sequência de células e lógica de recuperação/preparação/validação/dispatch do notebook Link2iOS. As alterações são específicas do Sparking! ZERO e da branch isolada `SpkIOSzero`.

**Dois caminhos:**
- **Retomar:** autenticar o Drive, clonar os scripts e executar a recuperação. Ela encontra somente manifests `SpkIOSzero_*` e retoma uma preparação SPK completa, sem baixar novamente base/NSPs nem repetir extração, rewrite ou shaders.
- **Primeira preparação:** baixar o 7z do jogo base pelo link assinado, extrair o único NSP base, ler `update.nsp`, exatamente 10 DLCs `.nsp` e `prod.keys` de `Drive/<pasta configurada>/spk`, preparar RomFS/AOC e publicar o handoff.

A pasta `spk` é a única origem de update, DLCs e chaves. Os arquivos de BOTW/Kleo2iOS e quaisquer outros itens da pasta compartilhada são ignorados. A entrada SPK esperada é exatamente 1 NSP base + 1 update NSP + 10 DLC NSPs.

**Velocidade/transferência:** sem percentuais estimados artificialmente. Transferências exibem progresso real, tamanho validado e tempo. Clone e autenticação não contam como download do jogo.

In [ ]:
import os, sys, argparse, json
from pathlib import Path
from getpass import getpass
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/spk-ios-zero")

auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive_api_client = build(
    "drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False
)
folder = drive_api_client.files().get(
    fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType"
).execute()
assert folder.get("mimeType") == "application/vnd.google-apps.folder", folder
spk_folder = next(
    (x for x in drive_api_client.files().list(
        q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
        pageSize=100, fields="files(id,name,mimeType)"
    ).execute().get("files", [])
     if x.get("mimeType") == "application/vnd.google-apps.folder"
     and x.get("name", "").lower() == "spk"),
    None,
)
assert spk_folder, "A pasta compartilhada precisa conter a subpasta spk."
print("Google Drive autenticado:", folder.get("name"))
print("Pasta exclusiva do Sparking! ZERO:", spk_folder.get("name"))
print("Autenticação concluída; nenhum jogo foi baixado nesta célula.")

## 1. Obter o pipeline SpkIOSzero

Clona somente a branch `SpkIOSzero`. O checkout pequeno dos scripts não é o download dos arquivos do jogo.

In [ ]:
%cd /content
!rm -rf /content/SpkIOSzero-repo
!git clone --depth 1 --branch SpkIOSzero https://github.com/joao9vegeto-stack/KintTany.git /content/SpkIOSzero-repo
!git -C /content/SpkIOSzero-repo rev-parse --short HEAD

## 2. Recuperar uma preparação SPK já concluída (sem refazer)

Execute esta célula primeiro ao retomar uma sessão interrompida ou quando a preparação já tiver sido concluída. Ela procura exclusivamente pastas `SpkIOSzero_*` dentro da pasta compartilhada, lê o `handoff.json` e recupera o estado SPK.

**Não baixa NSPs, não extrai conteúdo, não executa AOT rewrite nem shader split.** Pastas `Link2iOS_BOTW_*`, `Kleo2iOS_GameData_*` e arquivos de outros jogos são ignorados.

In [ ]:
%cd /content/SpkIOSzero-repo
import importlib, hashlib, subprocess
sys.path.insert(0, "/content/SpkIOSzero-repo/colab")
# Remove módulos em cache para impedir execução de código de outro checkout.
for module_name in ("source_spk", "prepare_spk", "run_spk"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
repo_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd="/content/SpkIOSzero-repo", text=True
).strip()
import source_spk
import prepare_spk
import run_spk
expected_source = Path("/content/SpkIOSzero-repo/colab/source_spk.py").resolve()
loaded_source = Path(source_spk.__file__).resolve()
assert loaded_source == expected_source, f"source_spk carregado de caminho inesperado: {loaded_source}"
source_text = loaded_source.read_text(encoding="utf-8")
assert "update.nsp" in source_text and "dlc" in source_text.lower() and "validate_nsp" in source_text, (
    "O checkout não contém o suporte de origem NSP esperado para o Sparking! ZERO. "
    f"HEAD={repo_head}; arquivo={loaded_source}"
)
assert "len(dlc_items) != 10" in source_text, "A validação de exatamente 10 DLC NSPs está ausente."
print("[SpkIOSzero] Commit do código:", repo_head)
print("[SpkIOSzero] Módulo de origem:", loaded_source)
print("[SpkIOSzero] SHA-256 source_spk.py:", hashlib.sha256(loaded_source.read_bytes()).hexdigest())
RECOVERED_HANDOFF = run_spk.recover(
    argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID),
    api=drive_api_client,
)
HANDOFF_READY = RECOVERED_HANDOFF is not None

## 3. Primeira preparação Linux (executar somente se não houver recuperação)

Esta é a etapa demorada. O Colab baixa o arquivo 7z do jogo base pelo link assinado e extrai o único NSP. Update, exatamente 10 DLCs NSP e `prod.keys` são lidos exclusivamente de `Drive/<pasta configurada>/spk`.

O NXNative lê metadados, extrai o jogo, executa o rewrite AArch64 e prepara RomFS/AOC. O decode Unreal/Oodle, shader split, compilação Metal, framework e Shared Shell ficam para o runner macOS da Route C. O Colab salva o workspace e publica um manifest de recuperação no Drive.

**Não execute esta célula se a etapa 2 recuperou uma preparação válida.**

In [ ]:
if HANDOFF_READY:
    print("Preparação SPK válida recuperada. Pulando download, extração, rewrite e shader split.")
else:
    root_items = run_spk.list_children(drive_api_client, DRIVE_FOLDER_ID)
    spk_folder = next(
        (x for x in root_items
         if x.get("mimeType") == "application/vnd.google-apps.folder"
         and x.get("name", "").lower() == "spk"),
        None,
    )
    assert spk_folder, "A pasta compartilhada precisa conter a subpasta spk."
    spk_items = run_spk.list_children(drive_api_client, spk_folder["id"])
    package_items = [
        x for x in spk_items
        if x.get("mimeType") != "application/vnd.google-apps.folder"
        and x.get("name", "").lower().endswith(".nsp")
    ]
    update_items = [
        x for x in package_items
        if x.get("name", "").lower() == "update.nsp"
    ]
    assert len(update_items) == 1, f"Esperado update.nsp em spk; encontrados {len(update_items)}."
    other_items = [x for x in package_items if x["id"] != update_items[0]["id"]]
    named_base = [x for x in other_items if x.get("name", "").lower().startswith("base")]
    if len(other_items) == 10:
        dlc_items, base_items = other_items, []
    elif len(other_items) == 11 and len(named_base) == 1:
        base_items = named_base
        dlc_items = [x for x in other_items if x["id"] != base_items[0]["id"]]
    else:
        raise AssertionError(
            f"Esperado update.nsp + 10 DLCs .nsp (e opcionalmente um base*.nsp); "
            f"encontrados {len(other_items)} NSPs adicionais."
        )
    assert len(dlc_items) == 10, f"Esperados exatamente 10 DLCs .nsp em spk; encontrados {len(dlc_items)}."
    keys_items = [x for x in spk_items if x.get("name", "").lower() == "prod.keys"]
    assert len(keys_items) == 1, "prod.keys precisa existir diretamente dentro de spk."
    cached_archive = (ROOT / "7z-temporary" / "sparking-zero-base.7z").is_file()
    if not base_items and not cached_archive:
        download_url = getpass("Cole o link assinado atualizado do 7z com o NSP base do Sparking! ZERO: ").strip()
        assert download_url.startswith(("https://", "http://")), "URL 7z válida obrigatória na primeira preparação."
        os.environ["SPK_BASE_ARCHIVE_URL"] = download_url
    else:
        os.environ.pop("SPK_BASE_ARCHIVE_URL", None)
        if cached_archive and not base_items:
            print("O 7z já existe nesta sessão do Colab; será reutilizado sem novo download.")
        elif base_items:
            print("NSP base já existe em spk; não haverá novo download do 7z.")
    if "download_url" in locals():
        del download_url
    run_spk.prepare(
        argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID),
        api=drive_api_client,
    )
    HANDOFF_PATH = Path("/content/spk-ios-zero-handoff.json")
    assert HANDOFF_PATH.is_file(), "A preparação terminou, mas o handoff local não foi criado."
    RECOVERED_HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
    HANDOFF_READY = (
        RECOVERED_HANDOFF.get("project") == "SpkIOSzero"
        and RECOVERED_HANDOFF.get("branch") == "SpkIOSzero"
        and RECOVERED_HANDOFF.get("title_id") == "010035F022078000"
        and RECOVERED_HANDOFF.get("source", {}).get("package_count") == 12
        and RECOVERED_HANDOFF.get("source", {}).get("dlc_nsp_count") == 10
    )
    assert HANDOFF_READY, "O handoff local não corresponde ao conjunto SPK (base + update + 10 DLC NSPs)."

## 4. Verificação do resultado e checkpoint de recuperação

O manifest persistido confirma Title ID, versão, hash do workspace, RomFS e imagens AOC extraídas. Também valida a origem: 1 NSP base + 1 update NSP + exatamente 10 DLCs NSP. Se o runtime do Colab reiniciar, basta repetir autenticação + clone + recuperação; não execute novamente a etapa longa.

In [ ]:
HANDOFF_PATH = Path("/content/spk-ios-zero-handoff.json")
assert HANDOFF_READY and HANDOFF_PATH.is_file(), "Nenhum handoff SPK recuperado/gerado."
HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
assert HANDOFF["title_id"] == "010035F022078000"
assert HANDOFF["branch"] == "SpkIOSzero"
assert HANDOFF["project"] == "SpkIOSzero"
assert HANDOFF["source"]["package_count"] == 12
assert HANDOFF["source"]["update_nsp_count"] == 1
assert HANDOFF["source"]["dlc_nsp_count"] == 10
assert HANDOFF["prepared"]["input_package_count"] == 12
assert HANDOFF["prepared"]["input_nsp_count"] == 12
assert HANDOFF["prepared"]["input_dlc_nsp_count"] == 10
assert HANDOFF["prepared"]["main_build_id"] == "68edafd68c7bc1a1af50b82aac838018"
assert HANDOFF["prepared"]["main_sha256"] == "a0f8f0644c92384ddb1baac1c21aecbeeace5bc99ad40895e216add61da78efe"
assert {x["name"] for x in HANDOFF["game_data"]["aoc"]} == {x["name"] for x in HANDOFF["prepared"]["aoc_images"]}
print("Sparking! ZERO — preparação verificada.")
print("Request:", HANDOFF["request_id"])
print("Drive:", HANDOFF.get("drive_output_folder_url"))
print("Workspace:", f'{HANDOFF["work_archive"]["size"]:,} bytes')
print("RomFS:", f'{HANDOFF["game_data"]["romfs"]["size"]:,} bytes')
print("DLC NSPs de origem:", HANDOFF["source"]["dlc_nsp_count"])
print("AOC images extraídas:", len(HANDOFF["game_data"]["aoc"]))
print("Pronto para o handoff do GitHub Actions.")

## 5. Última célula: enviar handoff e disparar somente SpkIOSzero

Use o Fine-grained PAT com `Contents: Read and write` restrito ao repositório. Esta célula envia apenas `handoff.json` para a branch `SpkIOSzero`; o workflow final baixa e valida os artefatos preparados no Drive. Não altera `main`, `Link2iOS` nem `Kleo2iOS`.

In [ ]:
%cd /content/SpkIOSzero-repo
!python colab/run_spk.py dispatch --handoff /content/spk-ios-zero-handoff.json